In [1]:
!pip install -q -U transformers trl peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 118.1 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.5 MB/s eta 0:00:00:00:0100:01


In [2]:
!nvidia-smi

Fri Aug  7 21:13:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

Mounted at /content/drive


In [10]:
cd /content/drive/MyDrive/Web-of-science-abstracts/train_pairs.jsonl

[Errno 20] Not a directory: '/content/drive/MyDrive/Web-of-science-abstracts/train_pairs.jsonl'
/content/drive/MyDrive/Web-of-science-abstracts


In [ ]:
# =============================================================================
# Step 5: QLoRA DPO Fine-Tuning of Qwen2.5-7B-Instruct on Colab
# =============================================================================
# Run this as a series of cells in a Colab notebook (Runtime > Change runtime
# type > GPU, T4 is fine for the free tier).
#
# Upload train_pairs.jsonl and validation_pairs.jsonl to the Colab file
# browser (left sidebar) before running, or mount Google Drive.
# =============================================================================

# ---- Cell 1: Install dependencies ----
# Run this first. Takes a few minutes. Restart runtime if prompted after.
"""
!pip install -q -U transformers trl peft accelerate bitsandbytes datasets
"""

# ---- Cell 2: Imports and config ----
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_PATH = "/content/drive/MyDrive/Web-of-science-abstracts/train_pairs.jsonl"
VAL_PATH = "/content/drive/MyDrive/Web-of-science-abstracts/validation_pairs.jsonl"
OUTPUT_DIR = "qwen25-7b-plantprotein-dpo"

# ---- Cell 3: Load and format data ----
# TRL's DPOTrainer expects a dataset with columns: prompt, chosen, rejected.
# Our jsonl already has these (plus meta_source/meta_trap_type, which the
# trainer will just ignore, no need to strip them).

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)

print(f"Train pairs: {len(train_data)}")
print(f"Validation pairs: {len(val_data)}")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# ---- Cell 4: Load tokenizer and 4-bit quantized model (QLoRA) ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # required for training

# ---- Cell 5: LoRA configuration ----
# Rank kept modest (r=16) given the small training set (~370 pairs) --
# a smaller rank reduces overfitting risk on limited data.
# target_modules covers Qwen2's attention + MLP projection layers.
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# ---- Cell 6: Training configuration ----
# Epochs kept low (3) given the small dataset -- watch the validation loss
# in the logs; if it starts rising while training loss keeps falling,
# that's overfitting, and you should stop earlier (reduce num_train_epochs
# and rerun, or use load_best_model_at_end with early stopping).
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 2*4 = 8
    gradient_checkpointing=True,     # saves memory, important on a T4
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    bf16=True,
    beta=0.1,                        # DPO temperature parameter, 0.1 is a
                                      # standard, reasonable default
    max_length=1024,                 # covers title+abstract+instruction+response;
                                      # controls truncation of the full prompt+completion
                                      # sequence combined (max_prompt_length was removed
                                      # in recent TRL versions -- filter/shorten overlong
                                      # prompts beforehand if needed instead)
    report_to="none",                # set to "wandb" if you use Weights & Biases
)

# ---- Cell 7: Build trainer and run ----
trainer = DPOTrainer(
    model=model,
    ref_model=None,                  # None lets TRL derive the reference
                                      # model from the base model + LoRA adapters
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

# ---- Cell 8: Save the fine-tuned LoRA adapters ----
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapters to {OUTPUT_DIR}")

# Optional: zip and download the adapter folder before your Colab session ends,
# Colab storage is NOT persistent across sessions unless you mount Drive.
# !zip -r qwen25-7b-plantprotein-dpo.zip {OUTPUT_DIR}
# from google.colab import files
# files.download("qwen25-7b-plantprotein-dpo.zip")

Train pairs: 369
Validation pairs: 64


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/369 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/369 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
20,0.000602,0.000098,1.853000,110841.000000,-2.081235,-1.929963,0.640764,6.943233,-3.086102,1.000000,10.029335,-119.526067,-128.679645
40,0.000013,0.000004,1.732758,228436.000000,-1.898928,-0.857918,0.667160,7.255843,-6.277705,1.000000,13.533547,-116.399968,-160.595672
60,0.000014,0.000003,1.712840,335164.000000,-1.882023,-0.722603,0.669251,7.253861,-6.613291,1.000000,13.867151,-116.419791,-163.951533


# checking word count of abstracts

In [13]:
"""
Check word and token counts for abstracts in your training data, to catch
anything that might get truncated by DPOConfig's max_length setting.

Run this against train_pairs.jsonl (or dpo_pairs.jsonl) before training.
"""

import json
import re

PATH = "/content/drive/MyDrive/Web-of-science-abstracts/train_pairs.jsonl"  # change to whatever file you want to check
MAX_LENGTH_TOKENS = 1024     # should match max_length in your DPOConfig

def extract_abstract(prompt: str) -> str:
    """Pull just the abstract text out of the full prompt string."""
    match = re.search(r"Abstract:\s*(.*)", prompt, re.DOTALL)
    return match.group(1).strip() if match else ""

def word_count(text: str) -> int:
    return len(text.split())

def rough_token_estimate(text: str) -> int:
    # ~0.75 words per token is a standard rough English estimate
    return round(word_count(text) / 0.75)

def main():
    with open(PATH) as f:
        pairs = [json.loads(line) for line in f]

    counts = []
    for p in pairs:
        abstract = extract_abstract(p["prompt"])
        wc = word_count(abstract)
        # rough estimate of the FULL sequence: instruction + title + abstract
        # + the chosen response text, since that's what max_length actually covers
        full_text = p["prompt"] + " " + p["chosen"]
        full_tokens_est = rough_token_estimate(full_text)
        counts.append((wc, full_tokens_est, p["prompt"][:80]))

    counts.sort(key=lambda x: -x[1])  # longest first

    print(f"Checked {len(counts)} entries from {PATH}\n")

    print("Top 10 longest (by estimated full sequence token count):")
    for wc, tok_est, preview in counts[:10]:
        flag = "  <-- OVER max_length, will be truncated" if tok_est > MAX_LENGTH_TOKENS else ""
        print(f"  abstract words: {wc:4d} | est. full tokens: {tok_est:4d}{flag}")
        print(f"    {preview}...")

    over_limit = [c for c in counts if c[1] > MAX_LENGTH_TOKENS]
    print(f"\n{len(over_limit)} of {len(counts)} entries estimated to exceed "
          f"max_length={MAX_LENGTH_TOKENS} tokens and may be truncated.")

    avg_wc = sum(c[0] for c in counts) / len(counts)
    avg_tok = sum(c[1] for c in counts) / len(counts)
    print(f"\nAverage abstract word count: {avg_wc:.0f}")
    print(f"Average estimated full sequence tokens: {avg_tok:.0f}")

if __name__ == "__main__":
    main()

Checked 369 entries from /content/drive/MyDrive/Web-of-science-abstracts/train_pairs.jsonl

Top 10 longest (by estimated full sequence token count):
  abstract words:  349 | est. full tokens:  573
    Classify this paper as relevant or irrelevant to plant protein functional proper...
  abstract words:  332 | est. full tokens:  568
    Classify this paper as relevant or irrelevant to plant protein functional proper...
  abstract words:  329 | est. full tokens:  553
    Classify this paper as relevant or irrelevant to plant protein functional proper...
  abstract words:  312 | est. full tokens:  536
    Classify this paper as relevant or irrelevant to plant protein functional proper...
  abstract words:  300 | est. full tokens:  525
    Classify this paper as relevant or irrelevant to plant protein functional proper...
  abstract words:  306 | est. full tokens:  520
    Classify this paper as relevant or irrelevant to plant protein functional proper...
  abstract words:  279 | est. full 